# RandomForest + XGBoost Training for Frontend-Only Inference

This notebook trains and evaluates RandomForest/XGBoost models using the existing project dataset, exports metrics/artifacts, and saves project-compatible model files used by the frontend ML bridge.

## 1) Environment Setup and Package Installation
Install/import required packages, set random seeds, and show versions.

In [1]:
import importlib
import json
import os
import random
import subprocess
import sys
from pathlib import Path

REQUIRED = [
    "pandas",
    "numpy",
    "scikit-learn",
    "xgboost",
    "matplotlib",
    "seaborn",
    "joblib",
]

missing = []
for pkg in REQUIRED:
    module_name = "sklearn" if pkg == "scikit-learn" else pkg
    try:
        importlib.import_module(module_name)
    except Exception:
        missing.append(pkg)

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
import sklearn
print("scikit-learn:", sklearn.__version__)
import xgboost
print("xgboost:", xgboost.__version__)

Python: 3.13.12
pandas: 3.0.1
numpy: 2.4.2
scikit-learn: 1.8.0
xgboost: 3.2.0


## 2) Load Dataset and Define Target
Load project dataset, inspect schema, define feature/label columns, and validate missing values.

In [2]:
PROJECT_ROOT = Path.cwd().resolve().parents[1]
DATA_PATH = PROJECT_ROOT / "ml" / "data" / "openfoodfacts_features.csv"
SAVED_MODELS_DIR = PROJECT_ROOT / "ml" / "saved_models"
ARTIFACTS_DIR = PROJECT_ROOT / "ml" / "saved_models" / "notebook_artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

NUMERIC_FEATURES = [
    "energy_kcal_100g",
    "fat_100g",
    "saturated_fat_100g",
    "sugars_100g",
    "salt_100g",
    "fiber_100g",
    "proteins_100g",
    "additives_n",
    "nova_group",
]

LABEL_COLUMNS = [
    "added_sugar",
    "artificial_color",
    "preservative",
    "emulsifier",
    "artificial_flavor",
    "caffeine",
    "palm_oil",
    "natural_protein_source",
    "fiber_source",
]

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))

X = df[NUMERIC_FEATURES].copy()
y = df[LABEL_COLUMNS].copy().astype(int)

print("Missing values (features):")
print(X.isna().sum())
print("\nLabel distribution (positive counts):")
print(y.sum().sort_values(ascending=False))

Dataset shape: (1012, 22)
Columns: ['code', 'product_name', 'ingredients_text', 'nutrition_grade', 'energy_kcal_100g', 'fat_100g', 'saturated_fat_100g', 'sugars_100g', 'salt_100g', 'fiber_100g', 'proteins_100g', 'additives_n', 'nova_group', 'added_sugar', 'artificial_color', 'preservative', 'emulsifier', 'artificial_flavor', 'caffeine', 'palm_oil', 'natural_protein_source', 'fiber_source']
Missing values (features):
energy_kcal_100g      0
fat_100g              0
saturated_fat_100g    0
sugars_100g           0
salt_100g             0
fiber_100g            0
proteins_100g         0
additives_n           0
nova_group            0
dtype: int64

Label distribution (positive counts):
added_sugar               228
artificial_flavor         104
artificial_color           89
natural_protein_source     88
fiber_source               83
emulsifier                 80
preservative               39
palm_oil                   18
caffeine                    7
dtype: int64


## 3) Preprocessing Pipeline (Numeric/Categorical)
Build a consistent preprocessing pipeline used by both models.

In [3]:
preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]),
            NUMERIC_FEATURES,
        )
    ],
    remainder="drop",
)

print("Preprocessing pipeline ready.")

Preprocessing pipeline ready.


## 4) Train/Validation/Test Split
Use multilabel stratification proxy via label cardinality bins and print expected split stats.

In [4]:
cardinality = y.sum(axis=1).astype(int)
strat_bins = np.where(cardinality >= 3, 3, cardinality)

X_train_full, X_test, y_train_full, y_test, strat_train_full, _ = train_test_split(
    X,
    y,
    strat_bins,
    test_size=0.15,
    random_state=SEED,
    stratify=strat_bins,
)

strat_train = np.where(y_train_full.sum(axis=1).astype(int) >= 3, 3, y_train_full.sum(axis=1).astype(int))
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.1765,  # ~15% of original after first split
    random_state=SEED,
    stratify=strat_train,
)

print("Split sizes")
print({"train": len(X_train), "val": len(X_val), "test": len(X_test)})
print("Train label ratio (mean positives per label):")
print(y_train.mean().round(4))

Split sizes
{'train': 708, 'val': 152, 'test': 152}
Train label ratio (mean positives per label):
added_sugar               0.2274
artificial_color          0.0833
preservative              0.0452
emulsifier                0.0791
artificial_flavor         0.0960
caffeine                  0.0056
palm_oil                  0.0212
natural_protein_source    0.0904
fiber_source              0.0791
dtype: float64


## 5) Train RandomForest Classifier
Train baseline/tuned RandomForest and save model.

In [5]:
rf_base = RandomForestClassifier(
    n_estimators=400,
    max_depth=12,
    min_samples_split=6,
    min_samples_leaf=3,
    class_weight="balanced_subsample",
    random_state=SEED,
    n_jobs=-1,
)

rf_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", MultiOutputClassifier(rf_base, n_jobs=-1)),
    ]
)

rf_pipeline.fit(X_train, y_train)
rf_path_notebook = SAVED_MODELS_DIR / "rf_pipeline_multilabel.joblib"
joblib.dump(rf_pipeline, rf_path_notebook)
print("Saved notebook RF pipeline:", rf_path_notebook)

Saved notebook RF pipeline: C:\Users\Shikhar\OneDrive\Desktop\sem4_projects\NutriScan\nutriscan-ai\ml\saved_models\rf_pipeline_multilabel.joblib


## 6) Train XGBoost Classifier
Train XGB in a multilabel wrapper with tuned hyperparameters and save model.

In [6]:
xgb_base = XGBClassifier(
    n_estimators=700,
    learning_rate=0.035,
    max_depth=3,
    min_child_weight=4,
    subsample=0.78,
    colsample_bytree=0.78,
    gamma=0.25,
    reg_lambda=2.8,
    reg_alpha=0.35,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=SEED,
    n_jobs=4,
)

xgb_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", MultiOutputClassifier(xgb_base, n_jobs=1)),
    ]
)

xgb_pipeline.fit(X_train, y_train)
xgb_path_notebook = SAVED_MODELS_DIR / "xgb_pipeline_multilabel.joblib"
joblib.dump(xgb_pipeline, xgb_path_notebook)
print("Saved notebook XGB pipeline:", xgb_path_notebook)

Saved notebook XGB pipeline: C:\Users\Shikhar\OneDrive\Desktop\sem4_projects\NutriScan\nutriscan-ai\ml\saved_models\xgb_pipeline_multilabel.joblib


## 7) Evaluate Models with Accuracy and Metrics
Compute accuracy, precision, recall, F1, ROC-AUC, confusion matrix, and classification reports.

In [7]:
def _predict_proba_multilabel(pipeline: Pipeline, X_input: pd.DataFrame) -> np.ndarray:
    probs = pipeline.predict_proba(X_input)
    out = []
    for p in probs:
        arr = np.asarray(p)
        if arr.ndim == 2 and arr.shape[1] > 1:
            out.append(arr[:, 1])
        else:
            out.append(arr.reshape(-1))
    return np.vstack(out).T


def evaluate_model(name: str, pipeline: Pipeline, X_eval: pd.DataFrame, y_eval: pd.DataFrame, threshold: float = 0.5):
    y_prob = _predict_proba_multilabel(pipeline, X_eval)
    y_pred = (y_prob >= threshold).astype(int)
    y_true = y_eval.values

    metrics = {
        "model": name,
        "subset_accuracy": float(accuracy_score(y_true, y_pred)),
        "micro_precision": float(precision_score(y_true, y_pred, average="micro", zero_division=0)),
        "micro_recall": float(recall_score(y_true, y_pred, average="micro", zero_division=0)),
        "micro_f1": float(f1_score(y_true, y_pred, average="micro", zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

    try:
        metrics["macro_roc_auc"] = float(roc_auc_score(y_true, y_prob, average="macro"))
    except Exception:
        metrics["macro_roc_auc"] = None

    print(f"\n{name} classification report:\n")
    print(classification_report(y_true, y_pred, target_names=LABEL_COLUMNS, zero_division=0))

    return metrics, y_true, y_pred, y_prob

rf_val_metrics, y_val_true, y_val_pred_rf, y_val_prob_rf = evaluate_model("RandomForest", rf_pipeline, X_val, y_val)
xgb_val_metrics, _, y_val_pred_xgb, y_val_prob_xgb = evaluate_model("XGBoost", xgb_pipeline, X_val, y_val)

comparison_df = pd.DataFrame([rf_val_metrics, xgb_val_metrics]).sort_values("micro_f1", ascending=False)
comparison_df


RandomForest classification report:

                        precision    recall  f1-score   support

           added_sugar       0.52      0.42      0.46        36
      artificial_color       0.69      0.60      0.64        15
          preservative       0.33      0.80      0.47         5
            emulsifier       0.25      0.50      0.33         8
     artificial_flavor       0.57      0.42      0.48        19
              caffeine       0.33      1.00      0.50         1
              palm_oil       0.00      0.00      0.00         0
natural_protein_source       0.38      0.25      0.30        12
          fiber_source       0.50      0.38      0.43        13

             micro avg       0.45      0.45      0.45       109
             macro avg       0.40      0.49      0.40       109
          weighted avg       0.50      0.45      0.46       109
           samples avg       0.22      0.23      0.21       109


XGBoost classification report:

                        precis

c:\Users\Shikhar\OneDrive\Desktop\sem4_projects\NutriScan\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
c:\Users\Shikhar\OneDrive\Desktop\sem4_projects\NutriScan\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


,model,subset_accuracy,micro_precision,micro_recall,micro_f1,macro_f1,macro_roc_auc
0,RandomForest,0.546053,0.449541,0.449541,0.449541,0.403105,NaN
1,XGBoost,0.618421,0.576923,0.275229,0.372671,0.257071,NaN


In [8]:
best_model_name = comparison_df.iloc[0]["model"]
best_pipeline = rf_pipeline if best_model_name == "RandomForest" else xgb_pipeline

# Confusion matrix on a representative label for visualization.
label_for_cm = "added_sugar"
label_idx = LABEL_COLUMNS.index(label_for_cm)
cm = confusion_matrix(y_val_true[:, label_idx], (y_val_prob_rf if best_model_name == "RandomForest" else y_val_prob_xgb)[:, label_idx] >= 0.5)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title(f"Confusion Matrix ({best_model_name}) - {label_for_cm}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
cm_path = ARTIFACTS_DIR / "confusion_matrix.png"
plt.tight_layout()
plt.savefig(cm_path, dpi=150)
plt.close()
print("Saved confusion matrix:", cm_path)

Saved confusion matrix: C:\Users\Shikhar\OneDrive\Desktop\sem4_projects\NutriScan\nutriscan-ai\ml\saved_models\notebook_artifacts\confusion_matrix.png


## 8) Test on a Secondary Dataset
Evaluate on an additional compatible dataset (external if available, otherwise sampled holdout-like set).

In [9]:
secondary_path = PROJECT_ROOT / "ml" / "data" / "openfoodfacts_features_secondary.csv"
if secondary_path.exists():
    df2 = pd.read_csv(secondary_path)
else:
    # Fallback secondary set from original data to validate reproducible generalization workflow.
    df2 = df.sample(frac=0.12, random_state=SEED + 11).copy()

X2 = df2[NUMERIC_FEATURES].copy()
y2 = df2[LABEL_COLUMNS].copy().astype(int)

secondary_metrics, y2_true, y2_pred, y2_prob = evaluate_model(f"{best_model_name}-secondary", best_pipeline, X2, y2)
pd.DataFrame([secondary_metrics])


RandomForest-secondary classification report:

                        precision    recall  f1-score   support

           added_sugar       0.81      0.76      0.79        29
      artificial_color       0.65      0.92      0.76        12
          preservative       0.78      1.00      0.88         7
            emulsifier       0.67      0.67      0.67         9
     artificial_flavor       0.90      0.53      0.67        17
              caffeine       0.00      0.00      0.00         1
              palm_oil       0.67      0.67      0.67         3
natural_protein_source       1.00      0.54      0.70        13
          fiber_source       0.78      0.78      0.78         9

             micro avg       0.77      0.71      0.74       100
             macro avg       0.69      0.65      0.66       100
          weighted avg       0.80      0.71      0.73       100
           samples avg       0.35      0.37      0.35       100



,model,subset_accuracy,micro_precision,micro_recall,micro_f1,macro_f1,macro_roc_auc
0,RandomForest-secondary,0.743802,0.771739,0.71,0.739583,0.655235,0.954519


## 9) Generate Expected Output Artifacts
Save metrics summary, confusion matrix, feature importance, and sample predictions.

In [10]:
metrics_summary = pd.DataFrame([rf_val_metrics, xgb_val_metrics, secondary_metrics])
metrics_summary_path = ARTIFACTS_DIR / "metrics_summary.csv"
metrics_summary.to_csv(metrics_summary_path, index=False)

# Feature importance from best model (average over multilabel estimators).
if best_model_name == "RandomForest":
    estimators = best_pipeline.named_steps["model"].estimators_
    fi = np.mean([est.feature_importances_ for est in estimators], axis=0)
else:
    estimators = best_pipeline.named_steps["model"].estimators_
    fi = np.mean([est.feature_importances_ for est in estimators], axis=0)

feature_importance_df = pd.DataFrame({"feature": NUMERIC_FEATURES, "importance": fi}).sort_values("importance", ascending=False)
feature_importance_path = ARTIFACTS_DIR / "feature_importance.csv"
feature_importance_df.to_csv(feature_importance_path, index=False)

plt.figure(figsize=(8, 4))
sns.barplot(data=feature_importance_df, x="importance", y="feature", orient="h")
plt.title(f"Feature Importance ({best_model_name})")
plt.tight_layout()
fi_plot_path = ARTIFACTS_DIR / "feature_importance.png"
plt.savefig(fi_plot_path, dpi=150)
plt.close()

sample_n = min(30, len(X_test))
sample_prob = _predict_proba_multilabel(best_pipeline, X_test.iloc[:sample_n])
sample_pred = (sample_prob >= 0.5).astype(int)
sample_predictions = X_test.iloc[:sample_n].copy()
sample_predictions["predicted_labels"] = [
    ",".join([LABEL_COLUMNS[i] for i, v in enumerate(row) if v == 1]) or "none"
    for row in sample_pred
]
sample_predictions["max_probability"] = sample_prob.max(axis=1)
sample_predictions_path = ARTIFACTS_DIR / "sample_predictions.csv"
sample_predictions.to_csv(sample_predictions_path, index=False)

print("Saved:")
print("-", metrics_summary_path)
print("-", cm_path)
print("-", feature_importance_path)
print("-", fi_plot_path)
print("-", sample_predictions_path)
metrics_summary

Saved:
- C:\Users\Shikhar\OneDrive\Desktop\sem4_projects\NutriScan\nutriscan-ai\ml\saved_models\notebook_artifacts\metrics_summary.csv
- C:\Users\Shikhar\OneDrive\Desktop\sem4_projects\NutriScan\nutriscan-ai\ml\saved_models\notebook_artifacts\confusion_matrix.png
- C:\Users\Shikhar\OneDrive\Desktop\sem4_projects\NutriScan\nutriscan-ai\ml\saved_models\notebook_artifacts\feature_importance.csv
- C:\Users\Shikhar\OneDrive\Desktop\sem4_projects\NutriScan\nutriscan-ai\ml\saved_models\notebook_artifacts\feature_importance.png
- C:\Users\Shikhar\OneDrive\Desktop\sem4_projects\NutriScan\nutriscan-ai\ml\saved_models\notebook_artifacts\sample_predictions.csv


,model,subset_accuracy,micro_precision,micro_recall,micro_f1,macro_f1,macro_roc_auc
0,RandomForest,0.546053,0.449541,0.449541,0.449541,0.403105,NaN
1,XGBoost,0.618421,0.576923,0.275229,0.372671,0.257071,NaN
2,RandomForest-secondary,0.743802,0.771739,0.710000,0.739583,0.655235,0.954519


## 10) Export Best Model for Frontend Inference
Export notebook best model plus project-compatible artifacts used by frontend_infer.py.

In [11]:
best_model_path = SAVED_MODELS_DIR / "best_pipeline_multilabel.joblib"
joblib.dump(best_pipeline, best_model_path)

metadata = {
    "feature_order": NUMERIC_FEATURES,
    "label_columns": LABEL_COLUMNS,
    "best_model_name": best_model_name,
    "threshold": 0.5,
    "notebook_model_path": str(best_model_path),
}
metadata_path = ARTIFACTS_DIR / "frontend_model_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

# Project-compatible export for frontend_infer.py
project_root_str = str(PROJECT_ROOT)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

from ml.ingredient_model.off_pipeline import train_models

project_export = train_models(df=df, output_dir=SAVED_MODELS_DIR, rf_search_iter=10)
print("Project-compatible model artifacts:")
print("-", project_export.rf_path)
print("-", project_export.xgb_path)
print("-", project_export.metrics_path)

print("Notebook metadata:")
print("-", metadata_path)
print("-", best_model_path)

Project-compatible model artifacts:
- C:\Users\Shikhar\OneDrive\Desktop\sem4_projects\NutriScan\nutriscan-ai\ml\saved_models\rf_feature_model.joblib
- C:\Users\Shikhar\OneDrive\Desktop\sem4_projects\NutriScan\nutriscan-ai\ml\saved_models\xgb_feature_model.joblib
- C:\Users\Shikhar\OneDrive\Desktop\sem4_projects\NutriScan\nutriscan-ai\ml\saved_models\training_metrics.json
Notebook metadata:
- C:\Users\Shikhar\OneDrive\Desktop\sem4_projects\NutriScan\nutriscan-ai\ml\saved_models\notebook_artifacts\frontend_model_metadata.json
- C:\Users\Shikhar\OneDrive\Desktop\sem4_projects\NutriScan\nutriscan-ai\ml\saved_models\best_pipeline_multilabel.joblib


## 11) Frontend Integration Contract (No Backend)
Define JSON contract and run a local frontend-style inference check using frontend_infer.py utilities.

In [12]:
# Ensure relative model paths in frontend_infer.py resolve exactly as they do in Next.js runtime.
os.chdir(PROJECT_ROOT)
project_root_str = str(PROJECT_ROOT)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

from ml.ingredient_model.frontend_infer import ml_predict_flags

request_schema = {
    "type": "object",
    "required": ["nutriments"],
    "properties": {
        "nutriments": {
            "type": "object",
            "properties": {
                "energy_kcal": {"type": "number"},
                "fat_100g": {"type": "number"},
                "saturated_fat_100g": {"type": "number"},
                "sugar_100g": {"type": "number"},
                "salt_100g": {"type": "number"},
                "fiber_100g": {"type": "number"},
                "additives_count": {"type": "number"},
                "nova_group": {"type": "number"},
            },
        }
    },
}

sample_payload = {
    "nutriments": {
        "energy_kcal": 420,
        "fat_100g": 16.5,
        "saturated_fat_100g": 7.2,
        "sugar_100g": 24.0,
        "salt_100g": 0.9,
        "fiber_100g": 2.3,
        "additives_count": 5,
        "nova_group": 4,
    }
}

response = ml_predict_flags(sample_payload["nutriments"])

contract_path = ARTIFACTS_DIR / "frontend_contract.json"
contract_path.write_text(
    json.dumps(
        {
            "request_schema": request_schema,
            "sample_request": sample_payload,
            "sample_response": response,
            "js_example": {
                "description": "Use local Next.js API route backed by runMlBridge (no external backend).",
                "snippet": "fetch('/api/analyze-food', { method: 'POST', headers: {'Content-Type': 'application/json'}, body: JSON.stringify({ barcode: '737628064502' }) })",
            },
        },
        indent=2,
    ),
    encoding="utf-8",
)

print("Sample frontend response:")
print(json.dumps(response, indent=2))
print("Saved contract:", contract_path)

print("\nArtifact existence check:")
for p in [
    SAVED_MODELS_DIR / "rf_feature_model.joblib",
    SAVED_MODELS_DIR / "xgb_feature_model.joblib",
    SAVED_MODELS_DIR / "training_metrics.json",
    metrics_summary_path,
    cm_path,
    feature_importance_path,
    sample_predictions_path,
    contract_path,
]:
    print(f"{p}:", p.exists())

Sample frontend response:
{
  "model_ready": true,
  "ml_feature_probabilities": {
    "added_sugar": 0.6757,
    "artificial_color": 0.512,
    "preservative": 0.4011,
    "emulsifier": 0.5634,
    "artificial_flavor": 0.3205,
    "caffeine": 0.0256,
    "palm_oil": 0.0612,
    "natural_protein_source": 0.1372,
    "fiber_source": 0.1183
  },
  "ml_flags": {
    "added_sugar": 1,
    "artificial_color": 1,
    "preservative": 0,
    "emulsifier": 1,
    "artificial_flavor": 0,
    "caffeine": 0,
    "palm_oil": 0,
    "natural_protein_source": 0,
    "fiber_source": 0
  }
}
Saved contract: C:\Users\Shikhar\OneDrive\Desktop\sem4_projects\NutriScan\nutriscan-ai\ml\saved_models\notebook_artifacts\frontend_contract.json

Artifact existence check:
C:\Users\Shikhar\OneDrive\Desktop\sem4_projects\NutriScan\nutriscan-ai\ml\saved_models\rf_feature_model.joblib: True
C:\Users\Shikhar\OneDrive\Desktop\sem4_projects\NutriScan\nutriscan-ai\ml\saved_models\xgb_feature_model.joblib: True
C:\Users\Sh

c:\Users\Shikhar\OneDrive\Desktop\sem4_projects\NutriScan\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
